# Final trial-level modeling dataset index

This notebook converts the artifact-rejected outputs into an auditable index for model development. It does **not** create train/test folds yet.

A run is eligible when it contains at least **10 retained left-hand trials and 10 retained right-hand trials** after artifact rejection. This prevents empty or severely depleted runs from entering a model while preserving every original decision in a separate audit table.

The final index maps each usable trial to:

- its cleaned tensor file;
- its exact row inside that tensor;
- participant, dataset, phase, run, and original trial number;
- left/right class and numeric label;
- artifact measurements; and
- retained class counts for its run.

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

def find_project_data(start=Path.cwd()):
    for parent in (start.resolve(), *start.resolve().parents):
        candidate = parent / 'data' / 'processed' / 'Signals'
        if candidate.is_dir():
            return parent / 'data'
    raise FileNotFoundError('Could not locate the project data directory.')

DATA_ROOT = find_project_data()
CLEAN_ROOT = DATA_ROOT / 'processed' / 'all_trials_time_frequency_artifact_rejected'
CLEAN_TENSOR_ROOT = CLEAN_ROOT / 'tensors_by_file'
QUALITY_PATH = CLEAN_ROOT / 'artifact_rejection_manifest.csv'
OUTPUT_ROOT = DATA_ROOT / 'processed' / 'modeling_index'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_INDEX_PATH = OUTPUT_ROOT / 'trial_modeling_index.csv'
RUN_ELIGIBILITY_PATH = OUTPUT_ROOT / 'run_eligibility.csv'
TRIAL_AUDIT_PATH = OUTPUT_ROOT / 'trial_inclusion_audit.csv'
PARTICIPANT_COVERAGE_PATH = OUTPUT_ROOT / 'participant_coverage.csv'
PARTICIPANT_SPLIT_PATH = OUTPUT_ROOT / 'participant_split_assignment.csv'
SPLIT_MODEL_INDEX_PATH = OUTPUT_ROOT / 'trial_modeling_index_with_split.csv'

MIN_RETAINED_PER_CLASS_PER_RUN = 10
EXPECTED_TENSOR_TAIL_SHAPE = (27, 23, 512)
LABEL_TO_ID = {'left': 0, 'right': 1}

quality = pd.read_csv(QUALITY_PATH)
required_columns = {
    'source_file', 'dataset', 'participant', 'run', 'phase', 'trial',
    'class_label', 'retained', 'rejection_reason', 'max_ptp_uv',
    'max_ptp_channel', 'min_ptp_uv', 'min_ptp_channel',
}
missing_columns = required_columns - set(quality.columns)
if missing_columns:
    raise ValueError(f'Artifact manifest is missing columns: {sorted(missing_columns)}')
if len(quality) != 19_040:
    raise ValueError(f'Expected 19,040 trial decisions, found {len(quality):,}.')

print(f'Artifact decisions loaded: {len(quality):,}')
print(f'Candidate recordings: {quality["source_file"].nunique():,}')

Artifact decisions loaded: 19,040
Candidate recordings: 476


## Calculate run eligibility

Eligibility uses only retained-trial counts. It does not use participant performance or model outcomes.

In [7]:
run_keys = ['source_file', 'dataset', 'participant', 'run', 'phase']

retained_by_class = (
    quality.pivot_table(
        index=run_keys, columns='class_label', values='retained',
        aggfunc='sum', fill_value=0,
    )
    .rename(columns={'left': 'retained_left', 'right': 'retained_right'})
    .reset_index()
)
original_by_class = (
    quality.pivot_table(
        index=run_keys, columns='class_label', values='trial',
        aggfunc='size', fill_value=0,
    )
    .rename(columns={'left': 'original_left', 'right': 'original_right'})
    .reset_index()
)
run_artifacts = (
    quality.groupby(run_keys)
    .agg(
        original_trials=('trial', 'size'),
        retained_trials=('retained', 'sum'),
        extreme_amplitude_trials=('extreme_amplitude', 'sum'),
        flat_electrode_trials=('flat_electrode', 'sum'),
        maximum_ptp_uv=('max_ptp_uv', 'max'),
    )
    .reset_index()
)

run_eligibility = (
    run_artifacts.merge(original_by_class, on=run_keys, validate='one_to_one')
    .merge(retained_by_class, on=run_keys, validate='one_to_one')
)
run_eligibility['rejected_trials'] = run_eligibility['original_trials'] - run_eligibility['retained_trials']
run_eligibility['run_eligible'] = (
    run_eligibility['retained_left'].ge(MIN_RETAINED_PER_CLASS_PER_RUN)
    & run_eligibility['retained_right'].ge(MIN_RETAINED_PER_CLASS_PER_RUN)
)
run_eligibility['run_exclusion_reason'] = np.where(
    run_eligibility['run_eligible'],
    '',
    f'fewer_than_{MIN_RETAINED_PER_CLASS_PER_RUN}_retained_trials_in_at_least_one_class',
)
run_eligibility['clean_tensor_relative_path'] = run_eligibility['source_file'].map(
    lambda source: (
        Path('processed') / 'all_trials_time_frequency_artifact_rejected' / 'tensors_by_file'
        / Path(source).with_suffix('').parent
        / f'{Path(source).stem}_trial_ersp_clean.npz'
    ).as_posix()
)

run_eligibility.to_csv(RUN_ELIGIBILITY_PATH, index=False)
display(run_eligibility['run_eligible'].value_counts().rename_axis('eligible').to_frame('runs'))
display(
    run_eligibility.loc[~run_eligibility['run_eligible'], [
        'source_file', 'retained_left', 'retained_right', 'rejected_trials', 'run_exclusion_reason'
    ]].sort_values(['retained_left', 'retained_right'])
)

,runs
eligible,
True,461
False,15


,source_file,retained_left,retained_right,rejected_trials,run_exclusion_reason
472,DATA C/C87/C87_R3_onlineT.gdf,0,0,40,fewer_than_10_retained_trials_in_at_least_one_...
474,DATA C/C87/C87_R5_onlineT.gdf,0,0,40,fewer_than_10_retained_trials_in_at_least_one_...
475,DATA C/C87/C87_R6_onlineT.gdf,0,0,40,fewer_than_10_retained_trials_in_at_least_one_...
473,DATA C/C87/C87_R4_onlineT.gdf,0,3,37,fewer_than_10_retained_trials_in_at_least_one_...
471,DATA C/C87/C87_R2_acquisition.gdf,1,1,38,fewer_than_10_retained_trials_in_at_least_one_...
470,DATA C/C87/C87_R1_acquisition.gdf,2,7,31,fewer_than_10_retained_trials_in_at_least_one_...
250,DATA A/A51/A51_R1_acquisition.gdf,4,4,32,fewer_than_10_retained_trials_in_at_least_one_...
465,DATA C/C86/C86_R2_acquisition.gdf,6,4,30,fewer_than_10_retained_trials_in_at_least_one_...
468,DATA C/C86/C86_R5_onlineT.gdf,6,7,27,fewer_than_10_retained_trials_in_at_least_one_...
254,DATA A/A51/A51_R5_onlineT.gdf,6,10,24,fewer_than_10_retained_trials_in_at_least_one_...


## Verify tensor alignment and assign tensor row indices

`tensor_row_index` is the row to load from `X` and `y` in the referenced `.npz`. Trial numbers and labels are checked against every cleaned tensor before the index is written.

In [8]:
audit = quality.copy()
audit['tensor_row_index'] = pd.Series(pd.NA, index=audit.index, dtype='Int64')
audit['clean_tensor_relative_path'] = audit['source_file'].map(
    run_eligibility.set_index('source_file')['clean_tensor_relative_path']
)

for number, (source_file, rows) in enumerate(audit.groupby('source_file', sort=True), start=1):
    relative_path = rows['clean_tensor_relative_path'].iloc[0]
    tensor_path = DATA_ROOT / relative_path
    if not tensor_path.is_file():
        raise FileNotFoundError(tensor_path)

    retained_rows = rows.loc[rows['retained']].sort_values('trial')
    with np.load(tensor_path, allow_pickle=False) as tensor:
        saved_trial_numbers = tensor['trial_numbers']
        saved_labels = tensor['y']
        if not np.array_equal(saved_trial_numbers, retained_rows['trial'].to_numpy()):
            raise ValueError(f'Trial-number mismatch in {relative_path}.')
        expected_labels = retained_rows['class_label'].map(LABEL_TO_ID).to_numpy()
        if not np.array_equal(saved_labels, expected_labels):
            raise ValueError(f'Class-label mismatch in {relative_path}.')

    audit.loc[retained_rows.index, 'tensor_row_index'] = np.arange(len(retained_rows))
    if number % 100 == 0 or number == run_eligibility.shape[0]:
        print(f'Validated tensor alignment: {number}/{run_eligibility.shape[0]} recordings')

assert audit.loc[audit['retained'], 'tensor_row_index'].notna().all()
assert audit.loc[~audit['retained'], 'tensor_row_index'].isna().all()
print('All cleaned tensor trial numbers and labels align with the artifact manifest.')

Validated tensor alignment: 100/476 recordings
Validated tensor alignment: 200/476 recordings
Validated tensor alignment: 300/476 recordings
Validated tensor alignment: 400/476 recordings
Validated tensor alignment: 476/476 recordings
All cleaned tensor trial numbers and labels align with the artifact manifest.


## Create the complete inclusion audit and final model index

In [9]:
run_lookup = run_eligibility.set_index('source_file')
audit['retained_left_in_run'] = audit['source_file'].map(run_lookup['retained_left'])
audit['retained_right_in_run'] = audit['source_file'].map(run_lookup['retained_right'])
audit['retained_trials_in_run'] = audit['source_file'].map(run_lookup['retained_trials'])
audit['run_eligible'] = audit['source_file'].map(run_lookup['run_eligible'])
audit['run_exclusion_reason'] = audit['source_file'].map(run_lookup['run_exclusion_reason'])
audit['use_for_model'] = audit['retained'] & audit['run_eligible']

def trial_exclusion_reason(row):
    reasons = []
    if not row.retained:
        reasons.append(row.rejection_reason if isinstance(row.rejection_reason, str) and row.rejection_reason else 'artifact_rejected')
    if not row.run_eligible:
        reasons.append(row.run_exclusion_reason)
    return '|'.join(reasons)

audit['model_exclusion_reason'] = audit.apply(trial_exclusion_reason, axis=1)
audit.to_csv(TRIAL_AUDIT_PATH, index=False)

model_columns = [
    'dataset', 'participant', 'run', 'phase', 'source_file', 'trial',
    'class_label', 'clean_tensor_relative_path', 'tensor_row_index',
    'max_ptp_uv', 'max_ptp_channel', 'min_ptp_uv', 'min_ptp_channel',
    'retained_left_in_run', 'retained_right_in_run', 'retained_trials_in_run',
]
model_index = audit.loc[audit['use_for_model'], model_columns].copy()
model_index['label_id'] = model_index['class_label'].map(LABEL_TO_ID).astype('int8')
model_index['tensor_row_index'] = model_index['tensor_row_index'].astype(int)
model_index.insert(
    0, 'sample_id',
    model_index['participant'] + '_R' + model_index['run'].astype(str)
    + '_T' + model_index['trial'].astype(str),
)
if model_index['sample_id'].duplicated().any():
    raise ValueError('sample_id is not unique.')
model_index.to_csv(MODEL_INDEX_PATH, index=False)

participant_coverage = (
    model_index.groupby(['dataset', 'participant'])
    .agg(
        eligible_runs=('source_file', 'nunique'),
        acquisition_trials=('phase', lambda values: int(values.eq('acquisition').sum())),
        online_trials=('phase', lambda values: int(values.eq('online').sum())),
        left_trials=('class_label', lambda values: int(values.eq('left').sum())),
        right_trials=('class_label', lambda values: int(values.eq('right').sum())),
        total_trials=('sample_id', 'size'),
    )
    .reset_index()
)
participant_coverage.to_csv(PARTICIPANT_COVERAGE_PATH, index=False)

print(f'Eligible runs: {run_eligibility["run_eligible"].sum():,}/{len(run_eligibility):,}')
print(f'Model trials: {len(model_index):,}')
print(f'Participants represented: {model_index["participant"].nunique():,}')
display(model_index.groupby(['phase', 'class_label']).size().rename('trials').reset_index())
display(participant_coverage.describe(include='all'))

Eligible runs: 461/476
Model trials: 17,439
Participants represented: 79


,phase,class_label,trials
0,acquisition,left,2950
1,acquisition,right,2940
2,online,left,5785
3,online,right,5764


,dataset,participant,eligible_runs,acquisition_trials,online_trials,left_trials,right_trials,total_trials
count,79,79,79.000000,79.000000,79.000000,79.000000,79.000000,79.000000
unique,3,79,NaN,NaN,NaN,NaN,NaN,NaN
top,DATA A,A1,NaN,NaN,NaN,NaN,NaN,NaN
freq,55,1,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,5.835443,74.556962,146.189873,110.569620,110.177215,220.746835
std,NaN,NaN,0.517151,10.290333,23.788640,15.820859,16.433049,32.023016
min,NaN,NaN,3.000000,25.000000,50.000000,41.000000,40.000000,81.000000
25%,NaN,NaN,6.000000,73.500000,147.000000,110.500000,111.000000,223.000000
50%,NaN,NaN,6.000000,79.000000,156.000000,117.000000,117.000000,233.000000
75%,NaN,NaN,6.000000,80.000000,159.000000,119.000000,119.000000,238.000000


## Load indexed samples and perform final spot checks

This demonstrates the exact loading operation a later dataset class will use. It checks samples from the beginning, middle, and end of the index without decompressing every 20 GiB tensor again.

In [10]:
def load_indexed_sample(index_row):
    tensor_path = DATA_ROOT / index_row.clean_tensor_relative_path
    with np.load(tensor_path, allow_pickle=False) as tensor:
        X = tensor['X'][int(index_row.tensor_row_index)]
        y = int(tensor['y'][int(index_row.tensor_row_index)])
    return X, y

check_positions = sorted({0, len(model_index) // 2, len(model_index) - 1})
check_results = []
for position in check_positions:
    row = model_index.iloc[position]
    X, y = load_indexed_sample(row)
    check_results.append({
        'sample_id': row.sample_id,
        'shape': X.shape,
        'finite': bool(np.isfinite(X).all()),
        'saved_label': y,
        'indexed_label': int(row.label_id),
    })
    assert X.shape == EXPECTED_TENSOR_TAIL_SHAPE
    assert np.isfinite(X).all()
    assert y == int(row.label_id)

display(pd.DataFrame(check_results))
print(f'Final model index: {MODEL_INDEX_PATH}')
print(f'Complete trial audit: {TRIAL_AUDIT_PATH}')
print(f'Run eligibility: {RUN_ELIGIBILITY_PATH}')
print(f'Participant coverage: {PARTICIPANT_COVERAGE_PATH}')

,sample_id,shape,finite,saved_label,indexed_label
0,A1_R2_T1,"(27, 23, 512)",True,1,1
1,A48_R3_T23,"(27, 23, 512)",True,1,1
2,C86_R6_T38,"(27, 23, 512)",True,0,0


Final model index: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/modeling_index/trial_modeling_index.csv
Complete trial audit: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/modeling_index/trial_inclusion_audit.csv
Run eligibility: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/modeling_index/run_eligibility.csv
Participant coverage: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/modeling_index/participant_coverage.csv


## Fixed participant-level train/validation/test split

Participants—not trials—are divided into 55 training, 12 validation, and 12 test participants. The split is deterministic (`random_state=42`) and stratified by Dataset A/B/C so all three source datasets remain represented in every partition. No participant performance or trial label is used to choose a partition.

In [11]:
RANDOM_STATE = 42
participants = participant_coverage[['dataset', 'participant']].drop_duplicates().sort_values('participant')
assert len(participants) == 79

# Dataset C has only five participants, so use explicit allocations that keep
# every source dataset represented in validation and test.
dataset_allocations = {
    'DATA A': {'train': 39, 'validation': 8, 'test': 8},
    'DATA B': {'train': 13, 'validation': 3, 'test': 3},
    'DATA C': {'train': 3, 'validation': 1, 'test': 1},
}
rng = np.random.default_rng(RANDOM_STATE)
split_frames = []
for dataset, allocation in dataset_allocations.items():
    dataset_participants = participants.loc[participants['dataset'].eq(dataset)].copy()
    if len(dataset_participants) != sum(allocation.values()):
        raise ValueError(f'Unexpected participant count for {dataset}: {len(dataset_participants)}')
    dataset_participants = dataset_participants.iloc[rng.permutation(len(dataset_participants))]
    start = 0
    for split, count in allocation.items():
        split_frames.append(dataset_participants.iloc[start:start + count].assign(split=split))
        start += count

participant_split = pd.concat(split_frames, ignore_index=True).sort_values(['split', 'dataset', 'participant'])

assert participant_split['participant'].nunique() == 79
assert not participant_split['participant'].duplicated().any()
assert participant_split['split'].value_counts().to_dict() == {
    'train': 55, 'validation': 12, 'test': 12,
}

participant_split = participant_split.merge(
    participant_coverage,
    on=['dataset', 'participant'],
    how='left',
    validate='one_to_one',
)
participant_split.to_csv(PARTICIPANT_SPLIT_PATH, index=False)

split_model_index = model_index.merge(
    participant_split[['dataset', 'participant', 'split']],
    on=['dataset', 'participant'],
    how='left',
    validate='many_to_one',
)
if split_model_index['split'].isna().any():
    raise ValueError('At least one indexed trial did not receive a split assignment.')
if split_model_index.groupby('participant')['split'].nunique().max() != 1:
    raise ValueError('Participant leakage detected across splits.')
split_model_index.to_csv(SPLIT_MODEL_INDEX_PATH, index=False)

participant_summary = (
    participant_split.groupby(['split', 'dataset'])
    .agg(participants=('participant', 'nunique'), trials=('total_trials', 'sum'))
    .reset_index()
)
trial_summary = (
    split_model_index.groupby(['split', 'phase', 'class_label'])
    .size().rename('trials').reset_index()
)
display(participant_summary)
display(trial_summary)
print('Participants per split:', participant_split['split'].value_counts().to_dict())
print('Trials per split:', split_model_index['split'].value_counts().to_dict())
print('Participant leakage check passed: every participant occurs in exactly one split.')
print(f'Participant assignments: {PARTICIPANT_SPLIT_PATH}')
print(f'Split trial index: {SPLIT_MODEL_INDEX_PATH}')

,split,dataset,participants,trials
0,test,DATA A,8,1836
1,test,DATA B,3,651
2,test,DATA C,1,226
3,train,DATA A,39,8753
4,train,DATA B,13,2856
5,train,DATA C,3,547
6,validation,DATA A,8,1643
7,validation,DATA B,3,703
8,validation,DATA C,1,224


,split,phase,class_label,trials
0,test,acquisition,left,449
1,test,acquisition,right,458
2,test,online,left,896
3,test,online,right,910
4,train,acquisition,left,2059
5,train,acquisition,right,2042
6,train,online,left,4041
7,train,online,right,4014
8,validation,acquisition,left,442
9,validation,acquisition,right,440


Participants per split: {'train': 55, 'test': 12, 'validation': 12}
Trials per split: {'train': 12156, 'test': 2713, 'validation': 2570}
Participant leakage check passed: every participant occurs in exactly one split.
Participant assignments: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/modeling_index/participant_split_assignment.csv
Split trial index: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/modeling_index/trial_modeling_index_with_split.csv


## Modeling handoff

`trial_modeling_index_with_split.csv` is the input catalog for fixed-split model development. Fit preprocessing and model parameters using `train`, use `validation` for model selection and early stopping, and leave `test` untouched until the model design is finalized. A separate participant-grouped cross-validation scheme can be added later for the final CSP/LDA comparison.